<a href="https://colab.research.google.com/github/maggoatt/Grounded-Text-Summarization-of-Research-Papers/blob/main/Evidence_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Evidence Retrieval

BM25 implementation for retrieving top matches

Author: Lawrence Zhou

**Installations and imports**

In [1]:
%pip -q install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [2]:
from rank_bm25 import BM25Okapi
import json, re

In [3]:
def tokenize(s: str):
    return re.findall(r"[a-z0-9]+", s.lower())

In [ ]:
with open("../../../data/253098895.json", "r") as file:
    paper = json.load(file)

## Preprocessing

In [6]:
section_info = []
raw_texts = []

for section in paper["sections"]:
    # Include the title to make retrieval better
    raw_text = f'{section["section_title"]} {section["text"]}'
    curr_section = {
        "corpusId": paper["corpusid"],
        "title": paper["title"],
        "section_title": section["section_title"],
        "text": section["text"]
    }
    raw_texts.append(raw_text)
    section_info.append(curr_section)

In [7]:
print(len(raw_texts))

22


In [8]:
tokenized_texts = [tokenize(raw_text) for raw_text in raw_texts]

## BM25

In [9]:
bm25 = BM25Okapi(tokenized_texts)

In [ ]:
with open("../../../summaries/253098895_bart_summary.txt", 'r') as f:
    summary = f.read()
summary = summary.split('.')
print(summary)

summary_scores = []

for sentence in summary:
    sum = bm25.get_scores(tokenize(sentence))
    summary_scores.append(sum)

['SPABERT is a LM built upon a pretrained BERT and trained to produce contextualized geo-entity representations given large geographic datasets', ' For a pivot, p, SPAberT first linearizes its neighboring geo-entitie names to form a BERT-compatible input sequence, called a pseudo sentence', ' The representations support various downstream applications, including contextualized Geo-entity classification', ' SPABERT performs the best for the 1:125K maps (30-CA) (Tab', ' 4) Our hypothesis is that the test maps contain denser geo-entities than other scales', ' We simulate omission using the OSM dataset by gradually removing random neighbors of a pivot entity', '']


In [15]:
summary_scores = summary_scores[0]
top_k = 5
top_idxs = summary_scores.argsort()[::-1][:top_k]

In [16]:
for i in top_idxs:
    section = section_info[int(i)]
    print("Corpus Id", section["corpusId"])
    print("Paper Title: ", section["title"])
    print("Section Title: ", section["section_title"])
    print("Section Score: ", float(summary_scores[i]))
    print("Section snippet: ", section["text"][:250], "...")

Corpus Id 253098895
Paper Title:  SpaBERT: A Pretrained Language Model from Geographic Data for Geo-Entity Representation
Section Title:  SPABERT
Section Score:  31.667304273779653
Section snippet:  SPABERT is a LM built upon a pretrained BERT and further trained to produce contextualized geo-entity representations given large geographic datasets. Fig. 1 shows the outline of SPABERT, with the details below. This section first presents the prelim ...
Corpus Id 253098895
Paper Title:  SpaBERT: A Pretrained Language Model from Geographic Data for Geo-Entity Representation
Section Title:  Introduction
Section Score:  17.40045513314233
Section snippet:  Interpreting human behaviors requires considering human activities and their surrounding environment. Looking at a stopping location, [ Speedway , ], 1 from a person's trajectory, we might assume that this person needs to use the location's amenities ...
Corpus Id 253098895
Paper Title:  SpaBERT: A Pretrained Language Model from Geographic D